In [1]:
from openai import OpenAI
import json, time
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
client = OpenAI()

def call_llm(prompt):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()


In [3]:

PRODUCT_CONFIGS = {

    "saas_crm": {
        "product_name": "CRM Platform",
        "categories"  : ["Billing", "Technical", "Feature Request", "Spam", "Other"],
        "urgency"     : ["High", "Medium", "Low"],
        "sla_hours"   : {"High": 2, "Medium": 8, "Low": 24},
        "examples"    : [
            {
                "subject": "SFDC not syncing",
                "body"   : "Salesforce stopped pulling leads.",
                "output" : "Technical"
            },
            {
                "subject": "Charged wrong amount",
                "body"   : "Billed $299 instead of $99.",
                "output" : "Billing"
            }
        ]
    },

    "fintech_payments": {
        "product_name": "Payment Gateway",
        "categories"  : ["Transaction Failed", "Settlement Delay", "KYC Issue", "Fraud Alert", "Other"],
        "urgency"     : ["Critical", "High", "Medium"],
        "sla_hours"   : {"Critical": 1, "High": 4, "Medium": 12},
        "examples"    : [
            {
                "subject": "UPI payment failed",
                "body"   : "UPI transaction deducted but not credited to merchant.",
                "output" : "Transaction Failed"
            },
            {
                "subject": "KYC rejected again",
                "body"   : "Documents rejected for 3rd time. Aadhaar + PAN submitted.",
                "output" : "KYC Issue"
            }
        ]
    },

    "edtech": {
        "product_name": "EdTech Learning Platform",
        "categories"  : ["Course Access", "Payment", "Technical", "Content Query", "Spam", "Other"],
        "urgency"     : ["High", "Medium", "Low"],
        "sla_hours"   : {"High": 4, "Medium": 12, "Low": 48},
        "examples"    : [
            {
                "subject": "Can't access purchased course",
                "body"   : "Paid for the Python course but can't open any videos.",
                "output" : "Course Access"
            },
            {
                "subject": "Certificate not generated",
                "body"   : "Completed the course 3 days ago. Certificate still pending.",
                "output" : "Technical"
            }
        ]
    }
}


In [9]:
config = PRODUCT_CONFIGS['edtech']
categories_str = "\n".join([f"  - {cat}" for cat in config['categories']])
print(categories_str) # categories_str

  - Course Access
  - Payment
  - Technical
  - Content Query
  - Spam
  - Other


In [10]:
def build_dynamic_prompt(product_key, subject, body):
    """
    Build a product-specific prompt from config.
    Zero hardcoding — everything comes from PRODUCT_CONFIGS.
    """
    config = PRODUCT_CONFIGS[product_key]

    # Build categories list dynamically
    categories_str = "\n".join([f"  - {cat}" for cat in config['categories']])

    # Build examples block dynamically
    examples_str = ""
    for i, ex in enumerate(config['examples'], 1):
        examples_str += f"""
EXAMPLE {i}:
Subject: {ex['subject']}
Body: {ex['body']}
Output: {ex['output']}
"""

    # Build SLA context
    sla_str = ", ".join([
        f"{k}: {v}hr response"
        for k, v in config['sla_hours'].items()
    ])

    return f"""CRITICAL: Return ONLY the category name.

You are an expert support email classifier for {config['product_name']}.

CATEGORIES (pick exactly one):
{categories_str}

URGENCY SLAs: {sla_str}

EXAMPLES FROM THIS PRODUCT:
{examples_str}

NOW CLASSIFY:
Subject: {subject}
Body: {body}

Return ONLY the category name."""

In [14]:
def classify_dynamic(product_key, email):
    prompt = build_dynamic_prompt(
        product_key,
        email['subject'],
        email['body']
    )
    return call_llm(prompt)

In [15]:
test_emails = [
    {"product": "saas_crm",        "subject": "Login broken",           "body": "Can't sign in since morning.",              "expected": "Technical"},
    {"product": "fintech_payments", "subject": "Payment not credited",   "body": "Customer paid but order not confirmed.",     "expected": "Transaction Failed"},
    {"product": "edtech",           "subject": "Video not playing",      "body": "Purchased course but videos won't load.",   "expected": "Course Access"},
]

In [16]:
for item in test_emails:
    result = classify_dynamic(item['product'],item)
    print(result)

Technical
Transaction Failed
Technical
